In [69]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader


board_size = 15 # 15 = mid


# Load training data from JSON file
def load_training_data(file_path):
    with open(file_path, "r") as file:
        data = json.load(file)
    return data

class HexDataset(Dataset):
    def __init__(self, file_path):
        self.data = load_training_data(file_path)
        self.board_size = board_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        # Convert list to numpy array and reshape to (3, board_size, board_size)
        player1_board = np.array(sample["Player1Board"]).reshape(1, self.board_size, self.board_size)
        player2_board = np.array(sample["Player2Board"]).reshape(1, self.board_size, self.board_size)
        player3_board = np.array(sample["Player3Board"]).reshape(1, self.board_size, self.board_size)

        # Stack into a (3, board_size, board_size) tensor
        board_tensor = torch.tensor(np.vstack([player1_board, player2_board, player3_board]), dtype=torch.float32)

        # Target value (scalar)
        root_value = torch.tensor([sample["RootValue"]], dtype=torch.float32)

        # Target policy (1D tensor of board_size * board_size)
        policy = torch.tensor(sample["MoveProbabilities"], dtype=torch.float32)

        return board_tensor, policy, root_value


In [70]:
class ThreePlayerHexNN(nn.Module):
    def __init__(self):
        super(ThreePlayerHexNN, self).__init__()
        self.board_size = board_size
        input_channels = 3  # One binary 2D array per player

        # Initial convolution layer
        self.conv1 = nn.Conv2d(input_channels, 256, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(256)

        # Residual blocks
        self.res_block1 = self._residual_block(256)
        self.res_block2 = self._residual_block(256)
        self.res_block3 = self._residual_block(256)
        self.res_block4 = self._residual_block(256)
        self.res_block5 = self._residual_block(256)

        # Policy Head
        self.policy_head = nn.Sequential(
            nn.Conv2d(256, 2, kernel_size=1),
            nn.BatchNorm2d(2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(2 * board_size * board_size, board_size * board_size),
            nn.Softmax(dim=-1)
        )

        # Value Head
        self.value_head = nn.Sequential(
            nn.Conv2d(256, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(board_size * board_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Tanh()
        )

    def _residual_block(self, channels):
        return nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.res_block1(x) + x
        x = self.res_block2(x) + x
        x = self.res_block3(x) + x
        x = self.res_block4(x) + x
        x = self.res_block5(x) + x
        
        policy = self.policy_head(x)
        value = self.value_head(x)
        return policy, value


In [71]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("CuDNN Version:", torch.backends.cudnn.version())
print("GPU Count:", torch.cuda.device_count())
print("Current GPU:", torch.cuda.current_device() if torch.cuda.is_available() else "No GPU detected")
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


PyTorch Version: 2.5.1+cu121
CUDA Available: True
CUDA Version: 12.1
CuDNN Version: 90100
GPU Count: 1
Current GPU: 0
GPU Name: NVIDIA GeForce GTX 1060 6GB


In [77]:
batch_size = 20
num_epochs = 200
learning_rate = 0.01

data_path = "V:/MFF/bakalarka/training_data/training_batch_20250211_210231.json"  # Replace with actual filename


# Create dataset and data loader
dataset = HexDataset(data_path)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model, optimizer, and loss functions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = ThreePlayerHexNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

policy_loss_fn = nn.CrossEntropyLoss()  # Policy loss
value_loss_fn = nn.MSELoss()            # Value loss

# Training Loop
for epoch in range(num_epochs):
    total_loss = 0
    for boards, policies, values in dataloader:
        boards, policies, values = boards.to(device), policies.to(device), values.to(device)

        optimizer.zero_grad()

        # Forward pass
        predicted_policies, predicted_values = model(boards)

        # Compute losses
        policy_loss = policy_loss_fn(predicted_policies, policies)
        value_loss = value_loss_fn(predicted_values.squeeze(), values)

        loss = policy_loss + value_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(dataloader):.4f}")

# Save the trained model
torch.save(model.state_dict(), "trained_three_player_hex.pth")
print("Model saved successfully!")


Using device: cuda


C:\Users\Mazi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([20, 1])) that is different to the input size (torch.Size([20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/200], Loss: 5.9822
Epoch [2/200], Loss: 6.4154
Epoch [3/200], Loss: 6.4227
Epoch [4/200], Loss: 6.4224
Epoch [5/200], Loss: 6.4214
Epoch [6/200], Loss: 6.4199
Epoch [7/200], Loss: 6.4211
Epoch [8/200], Loss: 6.4224
Epoch [9/200], Loss: 6.4151
Epoch [10/200], Loss: 6.4213
Epoch [11/200], Loss: 6.4029
Epoch [12/200], Loss: 6.0117
Epoch [13/200], Loss: 5.5068
Epoch [14/200], Loss: 5.4333
Epoch [15/200], Loss: 5.4317
Epoch [16/200], Loss: 5.4261
Epoch [17/200], Loss: 5.4153
Epoch [18/200], Loss: 5.4182
Epoch [19/200], Loss: 5.4118
Epoch [20/200], Loss: 5.4138
Epoch [21/200], Loss: 5.4105
Epoch [22/200], Loss: 5.4118
Epoch [23/200], Loss: 5.4103
Epoch [24/200], Loss: 5.4105
Epoch [25/200], Loss: 5.4104
Epoch [26/200], Loss: 5.4101
Epoch [27/200], Loss: 5.4105
Epoch [28/200], Loss: 5.4102
Epoch [29/200], Loss: 5.4103
Epoch [30/200], Loss: 5.4099
Epoch [31/200], Loss: 5.4100
Epoch [32/200], Loss: 5.4098
Epoch [33/200], Loss: 5.4097
Epoch [34/200], Loss: 5.4099
Epoch [35/200], Loss: 5

In [73]:
model.load_state_dict(torch.load("trained_three_player_hex.pth"))
model.eval()


C:\Users\Mazi\AppData\Local\Temp\ipykernel_852\125904352.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("trained_three_player_hex.pth")

ThreePlayerHexNN(
  (conv1): Conv2d(3, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (res_block1): Sequential(
    (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
  )
  (res_block2): Sequential(
    (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
  )
  (res_block3): Seq

In [74]:
# CHECK THE TRAINING DATA IS OK


import json
import numpy as np

with open(data_path, "r") as file:
    data = json.load(file)

# Function to dynamically calculate board size and visualize a single sample
def visualize_training_sample(sample):
    """ Dynamically computes board size and prints merged board representation. """

    # Calculate board size dynamically (since it's always square)
    total_cells = len(sample["Player1Board"])
    size = int(np.sqrt(total_cells))  # Assumes square board

    # Convert 1D lists back to 2D numpy arrays
    p1_board = np.array(sample["Player1Board"]).reshape((size, size))
    p2_board = np.array(sample["Player2Board"]).reshape((size, size))
    p3_board = np.array(sample["Player3Board"]).reshape((size, size))

    # Merge all boards into a single visual representation
    merged_board = np.zeros((size, size), dtype=int)
    merged_board[p1_board == 1] = 1  # Mark player 1 cells
    merged_board[p2_board == 1] = 2  # Mark player 2 cells
    merged_board[p3_board == 1] = 3  # Mark player 3 cells

    # Print the boards
    print(f"\n--- Player 1 Board ({size}x{size}) ---")
    print(p1_board)
    print(f"\n--- Player 2 Board ({size}x{size}) ---")
    print(p2_board)
    print(f"\n--- Player 3 Board ({size}x{size}) ---")
    print(p3_board)
    print(f"\n--- Merged Board ({size}x{size}) ---")
    print(merged_board)

    # Print additional metadata
    print("\nRoot Value:", sample["RootValue"])
    print("Move Probabilities:")
    print(np.array(sample["MoveProbabilities"]).reshape((size, size)))

# Show the first training sample
if data:
    print("\nLoaded Training Samples:", len(data))
    visualize_training_sample(data[0])  # Print the first sample
else:
    print("No training samples found in the file.")



Loaded Training Samples: 96

--- Player 1 Board (15x15) ---
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]

--- Player 2 Board (15x15) ---
[[0 0 0 0 0 0 0 0 1 1 1 1 1 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0